<a href="https://colab.research.google.com/github/ShahaanK/Relic/blob/main/Persona_Extraction_Tester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import json
from openai import OpenAI
from google.colab import userdata


In [18]:


# ==========================================
# CONFIGURATION & API SETUP
# ==========================================
# Use the same Llama 3.1 8B or 70B model for extraction, or upgrade
# to a larger reasoning model if you want deeper psychological analysis.

client = OpenAI(
    api_key=userdata.get('NVI-KEY'),
    base_url="https://integrate.api.nvidia.com/v1"
)
# For analysis, a larger model like 70B or Nemotron(nvidia/nemotron-4-340b-instruct) often performs better
MODEL_NAME = "nvidia/nemotron-3-super-120b-a12b"

TARGET_CHARACTER = "Robert" # Updated to match your JSON data exactly
INPUT_FILE = "../content/run_6_llama7b_clean_historical_chat_log.jsonl" # Updated to your file

# ==========================================
# EXTRACTION LOGIC
# ==========================================
def extract_persona(messages, target_name):
    # Prepare the chat log as a single string for analysis
    log_text = ""
    for msg in messages:
        log_text += f"[{msg['timestamp']}] {msg['sender']}: {msg['message']}\n"

    system_instruction = f"""You are an expert psychological profiler and linguist.
Analyze the following text message history and extract a highly detailed persona profile for the user named '{target_name}'.
You MUST output ONLY a valid JSON object. Do NOT include any conversational filler. Do NOT wrap it in markdown block quotes.

The JSON object must follow this exact structure:
{{
    "demographics": "Inferred age, occupation, relationship to the other person",
    "core_traits": ["List of 3-5 personality traits"],
    "texting_quirks": ["List of 3-5 specific linguistic habits, punctuation styles, or emoji use"],
    "common_topics": ["List of topics this person frequently brings up"],
    "emotional_tone": "A brief description of their underlying emotional state and how they express affection or stress"
}}"""

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_instruction},
                {"role": "user", "content": f"Chat Log:\n{log_text}"}
            ],
            temperature=0.2, # Low temperature for analytical consistency
            # If the API endpoint supports strict JSON enforcement, this helps:
            # response_format={"type": "json_object"}
        )

        raw_output = response.choices[0].message.content.strip()

        # --- ROBUST JSON CLEANING ---
        # 1. Strip markdown code blocks if the model included them
        if raw_output.startswith("```json"):
            raw_output = raw_output[7:]
        elif raw_output.startswith("```"):
            raw_output = raw_output[3:]

        if raw_output.endswith("```"):
            raw_output = raw_output[:-3]

        # 2. Find the first '{' and last '}' to strip conversational filler
        start_idx = raw_output.find('{')
        end_idx = raw_output.rfind('}')

        if start_idx != -1 and end_idx != -1:
            raw_output = raw_output[start_idx:end_idx + 1]

        return raw_output.strip()

    except Exception as e:
        print(f"Extraction Error: {e}")
        return None

# ==========================================
# MAIN EXECUTION
# ==========================================
if __name__ == "__main__":
    print("Raw Data Loaded. Extracting Persona...")

    # Load the JSONL file
    chat_history = []
    try:
        with open(INPUT_FILE, 'r', encoding='utf-8') as f:
            for line in f:
                if line.strip():
                    chat_history.append(json.loads(line))
    except FileNotFoundError:
        print(f"Error: Could not find file '{INPUT_FILE}'")
        exit()

    print(f"Analyzing {len(chat_history)} messages...")

    profile_json_str = extract_persona(chat_history, TARGET_CHARACTER)

    print("\n=== EXTRACTED PERSONA PROFILE ===")
    if profile_json_str:
        try:
            # Pretty print the JSON output
            parsed_profile = json.loads(profile_json_str)
            print(json.dumps(parsed_profile, indent=4))
        except json.JSONDecodeError as e:
            # FALLBACK: If it STILL fails to parse, print the raw output so we can debug it
            print("Failed to parse JSON. Printing raw model output instead:\n")
            print(profile_json_str)
    else:
        print("Failed to generate a profile.")

Raw Data Loaded. Extracting Persona...
Analyzing 91 messages...

=== EXTRACTED PERSONA PROFILE ===
{
    "demographics": "Likely a middle-aged to older male, father of Alex, possibly employed in a maintenance or handyman role, living in a suburban home.",
    "core_traits": [
        "overprotective",
        "detail-oriented",
        "anxious",
        "caring",
        "persistent"
    ],
    "texting_quirks": [
        "Frequent use of ellipsis (...) to soften statements",
        "Repeated sign\u2011off 'Love, Dad' at end of messages",
        "Overuse of apologies and qualifiers like 'sorry', 'okay okay'",
        "Tendency to insert 'btw' and justifications before shifting topics",
        "Repetitive phrasing (e.g., 'just got', 'hope mom is okay')"
    ],
    "common_topics": [
        "Lawn maintenance",
        "Car oil/battery/mechanical issues",
        "Mom's health and hospital visits",
        "Reminders/chores",
        "Apologizing and seeking reassurance"
    ],
    "